# Chapter 2 — Yields, delays and the [α/Fe] knee

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/chemical-evolution-tutorial/blob/main/notebooks/02_yields_delays.ipynb)

**Goal.** Add iron to the one-zone model of Chapter 1. Core-collapse supernovae (CCSNe) make α-elements and some
iron within a few Myr of star formation; type Ia supernovae (SNe Ia) make mostly iron, with delays of 0.1 to
10 Gyr. The competition between the two produces the shape of every track in Chapter 0: a high [α/Fe] *plateau*,
a *knee* where SNe Ia start to matter, and a decline.

**You will learn**
1. what a *yield* is, and where the plateau value comes from;
2. what a *delay-time distribution* (DTD) is, and how to compute the SN Ia rate from a star-formation history;
3. why the [Fe/H] of the knee measures how efficiently a galaxy formed stars;
4. what sets [α/Fe] long after the knee.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# --- Data access (identical in every chapter) -------------------------------------------------------------
DATA_URL = 'https://github.com/vasilybelokurov/chemical-evolution-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def fetch(name, cache=Path('data')):
    """Return the local path of data file `name`, downloading it if needed; its SHA-256 must match the manifest."""
    want = MANIFEST['files'][name]['sha256']
    path = Path('../data_build') / name
    if not path.exists():
        path = cache / name
        if not path.exists() or sha256(path) != want:
            cache.mkdir(exist_ok=True)
            print(f'downloading {name} ...')
            urllib.request.urlretrieve(DATA_URL + name, path)
    assert sha256(path) == want, f'{name}: checksum mismatch'
    return path

## 2.1 Yields

A **yield** is the mass of an element that a population of stars returns as *new* material, per unit mass of stars
formed. We use the yield tables of H21's models (from the flexCE code: Limongi & Chieffi CCSNe, Iwamoto et al. W70
SNe Ia), integrated over a Kroupa initial mass function and tabulated against the metallicity $Z$ of the stars.

In [ ]:
Y = np.load(fetch('h21_ssp_bundle.npz'))
Zgrid = Y['z']

fig, ax = plt.subplots(figsize=(7, 4.3))
for el, col in (('mg', '#2a78d6'), ('si', '#1baf7a'), ('fe', '#eb6834')):
    ax.plot(Zgrid, Y[f'cc_{el}'], color=col, lw=2, label=f'{el.capitalize()}, CCSNe')
    ax.plot(Zgrid, Y[f'ia_{el}'], color=col, lw=2, ls='--', label=f'{el.capitalize()}, SNe Ia')
ax.set(xscale='log', yscale='log', xlabel='metallicity of the stars, Z', ylabel=r'yield [$M_\odot$ per $M_\odot$ formed]',
       ylim=(1e-5, 5e-3))
ax.legend(frameon=False, fontsize=8, ncol=2)
ax.grid(color='0.9', lw=0.6, which='both')

# constant yields used in this chapter: the table values at Z = 0.002 (flexCE's upper-bin lookup)
i = min(len(Zgrid) - 1, int(np.searchsorted(Zgrid, 0.002, side='right')))
YIELD = {el: (float(Y[f'cc_{el}'][i]), float(Y[f'ia_{el}'][i])) for el in ('mg', 'si', 'fe')}
for el, (cc, ia) in YIELD.items():
    print(f'{el}: CCSN {cc:.3e}   SN Ia {ia:.3e}')

**Figure 2.1.** Yields per unit mass of stars formed, against the stars' metallicity $Z$, for Mg (blue), Si
(green) and Fe (orange); solid: CCSNe, dashed: SNe Ia (H21's yield set).

CCSNe make Mg, Si and Fe in similar amounts, about $10^{-3}$ of the mass formed. SNe Ia make 1.6 times more iron than
CCSNe, a third as much Si, and almost no Mg. All yields change by less than 30% over two decades in $Z$, so in this
chapter we hold them fixed at their values at $Z = 0.002$ (printed above). Chapter 3 lets them vary.

**Solar scale.** [X/Fe] compares number ratios with the Sun's: $[{\rm X/Fe}] = \log_{10}(N_X/N_{\rm Fe}) -
\log_{10}(N_X/N_{\rm Fe})_\odot$. We use H21's solar abundances (Lodders 2003; $\log\epsilon$ = 7.47 for Fe, 7.54
for Si, 7.55 for Mg on the scale where hydrogen is 12) and convert masses to numbers with atomic masses 56, 28 and 24.

**The plateau.** Before the first SNe Ia, all iron and all Mg come from CCSNe, so the gas has exactly the CCSN yield
ratio, whatever the star-formation history:

$$[{\rm X/Fe}]_{\rm plateau} = \log_{10}\frac{y^{\rm cc}_X/A_X}{y^{\rm cc}_{\rm Fe}/A_{\rm Fe}} - (\log\epsilon_X - \log\epsilon_{\rm Fe})_\odot .$$

In [ ]:
LOGEPS = {'fe': 7.47, 'si': 7.54, 'mg': 7.55}     # Lodders (2003), as in H21 / flexCE
A = {'fe': 56, 'si': 28, 'mg': 24}
X_H = 0.73                                          # hydrogen mass fraction of the gas (held fixed)


def xfe_from_yields(el, y_el, y_fe):
    """[X/Fe] of gas whose X and Fe came in the mass ratio y_el : y_fe."""
    return np.log10((y_el / A[el]) / (y_fe / A['fe'])) - (LOGEPS[el] - LOGEPS['fe'])


plateau = {el: xfe_from_yields(el, YIELD[el][0], YIELD['fe'][0]) for el in ('mg', 'si')}
print({el: round(v, 3) for el, v in plateau.items()})

The model predicts plateaus of [Mg/Fe] = +0.26 and [Si/Fe] = +0.23. GSE, the most α-rich galaxy of Chapter 0, sits
at [Mg/Fe] ≈ +0.27–0.30 and [Si/Fe] ≈ +0.25 at [Fe/H] = −2: close to these values, though no galaxy in our sample
shows a flat plateau over a wide range of [Fe/H].

## 2.2 The delay-time distribution

A population of stars formed at time $t'$ produces SNe Ia at a rate $f(t-t')$ per unit mass formed. This
**delay-time distribution** is normalised so that $\int_0^\infty f(\tau)\,d\tau = 1$; the total iron per unit
mass is in the yield $y^{\rm Ia}_{\rm Fe}$. H21 use an exponential DTD with a minimum delay $t_D$ = 0.15 Gyr and a
time scale $\tau_{\rm Ia}$ = 1.5 Gyr. A common alternative, closer to what supernova surveys measure, is a power
law $f \propto \tau^{-1.1}$ beyond $t_D$. Its integral diverges only logarithmically slowly, and most of it lies
beyond the age of the Universe, so it is normalised over the delays that can occur, $t_D \le \tau \le 13.7$ Gyr.

The SN Ia "rate" (in units of the yield) of a galaxy is the star-formation history *convolved* with the DTD:

$$R_{\rm Ia}(t) = \int_0^t \dot M_\star(t')\,f(t-t')\,dt' .$$

In [ ]:
def dtd_exp(tau, t_d=0.15, tau_ia=1.5):
    """Exponential DTD with minimum delay t_d [Gyr], normalised to unit integral."""
    return np.where(tau >= t_d, np.exp(-(tau - t_d) / tau_ia) / tau_ia, 0.0)


def dtd_plaw(tau, t_d=0.15, slope=1.1, t_max=13.7):
    """Power-law DTD tau^-slope for t_d <= tau <= t_max, normalised to unit integral over that range."""
    tau = np.asarray(tau, float)
    norm = (slope - 1) / (t_d ** (1 - slope) - t_max ** (1 - slope))   # 1 / integral_{t_d}^{t_max} tau^-slope
    return np.where((tau >= t_d) & (tau <= t_max), norm * np.clip(tau, t_d, t_max) ** -slope, 0.0)


tau = np.linspace(0, 100, 1_000_001)                        # 0.1-Myr grid out to 100 Gyr, for the integrals
frac_1gyr = {}
for name, f in (('exponential', dtd_exp), ('power law', dtd_plaw)):
    total = np.trapezoid(f(tau), tau)
    frac_1gyr[name] = np.trapezoid(f(tau[tau <= 1]), tau[tau <= 1])
    print(f'{name:12s} integral = {total:.4f};  fraction within 1 Gyr = {frac_1gyr[name]:.3f}')
    assert abs(total - 1) < 2e-3                              # identity: normalisation (grid error ~1e-4)
assert round(frac_1gyr['exponential'], 2) == 0.43 and round(frac_1gyr['power law'], 2) == 0.48   # quoted below

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
tt = np.linspace(0, 13.7, 2000)
for name, f, col in (('exponential (H21)', dtd_exp, '#2a78d6'), (r'power law $\tau^{-1.1}$', dtd_plaw, '#eb6834')):
    axes[0].plot(tt, f(tt), color=col, lw=2, label=name)
    axes[1].plot(tt, np.cumsum(f(tt)) * (tt[1] - tt[0]), color=col, lw=2, label=name)
axes[0].set(xlabel=r'delay $\tau$ [Gyr]', ylabel=r'$f(\tau)$ [Gyr$^{-1}$]', yscale='log', ylim=(1e-3, 10))
axes[1].set(xlabel=r'delay $\tau$ [Gyr]', ylabel='fraction of SNe Ia exploded')
for ax in axes:
    ax.legend(frameon=False)
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()

**Figure 2.2.** Left: the two delay-time distributions, $f(\tau)$ in Gyr$^{-1}$ (log scale), both with a minimum
delay of 0.15 Gyr: exponential with $\tau_{\rm Ia}$ = 1.5 Gyr (blue, H21) and a power law $\tau^{-1.1}$ (orange).
Right: the fraction of all SNe Ia that have exploded by delay $\tau$.

The two shapes distribute the explosions differently: the exponential sets off 43% of its SNe Ia within 1 Gyr, the
power law 48% (printed above). The power law has more very short delays (its rate is highest just after $t_D$) and a
long tail that keeps exploding for the whole age of the Universe.

## 2.3 The model with iron

Each element now has a CCSN term, proportional to the current star formation, and an SN Ia term, proportional to the
convolution above. Everything else is Chapter 1:

$$\dot M_X = y^{\rm cc}_X\,\dot M_\star + y^{\rm Ia}_X\,R_{\rm Ia}(t) - (1+\eta-r)\,\dot M_\star\,\frac{M_X}{M_g}.$$

The convolution is a sum over all earlier steps: `np.dot(sfr[:i+1], f[i::-1])` multiplies the star formation at
each earlier step by the DTD at the corresponding delay (`f[i::-1]` is the DTD array reversed, so that the oldest
step meets the longest delay).

In [ ]:
def one_zone_fe(t_end, dt, inflow, tau_star, eta, r=0.4, Mg0=0.0, dtd=dtd_exp, yields=YIELD):
    """Chapter 1's Euler integration with Mg, Si, Fe from CCSNe (instantaneous) and SNe Ia (DTD convolution).
    Returns t [Gyr], gas Mg [Msun], SFR [Msun/Gyr], [Fe/H], and {'mg': [Mg/Fe], 'si': [Si/Fe]}."""
    n = int(round(t_end / dt))
    t = np.arange(n + 1) * dt
    Mg = np.zeros(n + 1)
    Mg[0] = Mg0
    M = {el: np.zeros(n + 1) for el in yields}
    sfr = np.zeros(n + 1)
    f = dtd(t) * dt                                          # fraction of SNe Ia in each step of delay
    for i in range(n):
        sfr[i] = Mg[i] / tau_star
        ia = np.dot(sfr[:i + 1], f[i::-1])                   # R_Ia(t_i): convolution of the SFR with the DTD
        loss = (1 + eta - r) * sfr[i]
        for el, (y_cc, y_ia) in yields.items():
            z = M[el][i] / Mg[i] if Mg[i] > 0 else 0.0
            M[el][i + 1] = M[el][i] + dt * (y_cc * sfr[i] + y_ia * ia - loss * z)
        Mg[i + 1] = Mg[i] + dt * (inflow(t[i]) - loss)
    sfr[n] = Mg[n] / tau_star
    with np.errstate(divide='ignore', invalid='ignore'):    # log of 0 at t = 0 gives -inf: harmless
        feh = np.log10(M['fe'] / Mg / A['fe'] / X_H) - (LOGEPS['fe'] - 12)
        xfe = {el: np.log10((M[el] / A[el]) / (M['fe'] / A['fe'])) - (LOGEPS[el] - LOGEPS['fe']) for el in ('mg', 'si')}
    return t, Mg, sfr, feh, xfe


def constant_sfr(tau_star, eta, t_end=13.7, dt=0.002, Mg0=1e9, **kw):
    """A galaxy with constant star formation: the inflow replaces exactly what is lost."""
    return one_zone_fe(t_end, dt, lambda t: (1 + eta - 0.4) * Mg0 / tau_star, tau_star, eta, Mg0=Mg0, **kw)


t, Mg, sfr, feh, xfe = constant_sfr(tau_star=2.0, eta=2.5)
late = {el: xfe_from_yields(el, sum(YIELD[el]), sum(YIELD['fe'])) for el in ('mg', 'si')}
early = t < 0.15
print('plateau  model:', {el: round(float(xfe[el][1]), 4) for el in xfe}, ' yields:', {el: round(v, 4) for el, v in plateau.items()})
print('late     model:', {el: round(float(xfe[el][-1]), 4) for el in xfe}, ' yields:', {el: round(v, 4) for el, v in late.items()})
for el in ('mg', 'si'):
    assert np.allclose(xfe[el][1:][early[1:]], plateau[el], atol=1e-10)   # identity: no SNe Ia before t_D
    assert abs(xfe[el][-1] - late[el]) < 0.005                           # limit: late equilibrium, constant SFR

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for el, col in (('mg', '#2a78d6'), ('si', '#1baf7a')):
    axes[0].plot(t[1:], xfe[el][1:], color=col, lw=2, label=f'[{el.capitalize()}/Fe]')
    axes[1].plot(feh[1:], xfe[el][1:], color=col, lw=2, label=f'[{el.capitalize()}/Fe]')
    for ax, x0 in ((axes[0], 0.0), (axes[1], -3)):
        ax.axhline(plateau[el], color=col, lw=0.8, ls=':')
        ax.axhline(late[el], color=col, lw=0.8, ls='--')
axes[0].axvline(0.15, color='0.5', lw=0.8)
axes[0].set(xscale='log', xlim=(0.01, 13.7), xlabel='time [Gyr]', ylabel='[X/Fe]')
axes[1].set(xlim=(-3, 0.3), xlabel='[Fe/H]', ylabel='[X/Fe]')
for ax in axes:
    ax.legend(frameon=False)
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()

**Figure 2.3.** A galaxy with constant star formation ($\tau_\star$ = 2 Gyr, $\eta$ = 2.5, exponential DTD).
[Mg/Fe] (blue) and [Si/Fe] (green) against time (left, log scale; grey line: the minimum SN Ia delay of 0.15 Gyr)
and against [Fe/H] (right). Dotted: the plateau from the CCSN yields; dashed: the late-time value from the summed
CCSN + SN Ia yields.

The model reproduces both limits exactly: the plateau until 0.15 Gyr, and, once the SN Ia rate has caught up with
the constant star formation, the ratio of total yields, [Mg/Fe] = −0.14 and [Si/Fe] = −0.07. [Si/Fe] falls less
than [Mg/Fe] because SNe Ia make some Si. In time, the decline is spread over 0.15 to ~5 Gyr; in [Fe/H] it is
squeezed into the last dex, from −1.2 to −0.1, because the gas approaches its equilibrium metallicity (Chapter 1)
while SNe Ia keep adding iron.

## 2.4 Where the knee sits

Define the **knee** as the point where [Si/Fe] has fallen 0.05 dex below the plateau. For constant star formation, the
SN Ia rate relative to the star-formation rate depends only on the DTD, so the knee occurs at about the same *time*
in every galaxy. But the *metallicity* reached by then depends on how fast stars formed: early on, before outflows
and dilution matter, the gas gains iron at the rate $y\,\dot M_\star/M_g = y/\tau_\star$, so
$Z_{\rm Fe}(t_{\rm knee}) \approx y\,t_{\rm knee}/\tau_\star$. Doubling $\tau_\star$ should lower the knee's [Fe/H]
by $\log_{10} 2 = 0.30$ dex, as long as the knee comes well before the gas depletion time. To isolate this, we
switch the wind off ($\eta$ = 0), so that $\tau_{\rm dep} = \tau_\star/0.6$ is long.

In [ ]:
taus = np.array([1.0, 2.0, 4.0, 8.0, 16.0])
knee_feh, knee_t, tracks = [], [], []
for ts in taus:
    t, Mg, sfr, feh, xfe = constant_sfr(tau_star=ts, eta=0.0, t_end=6.0)
    k = np.argmax(xfe['si'] < plateau['si'] - 0.05)          # first step 0.05 dex below the plateau
    knee_feh.append(feh[k]); knee_t.append(t[k]); tracks.append((feh, xfe['si']))
knee_feh, knee_t = np.array(knee_feh), np.array(knee_t)
step = np.diff(knee_feh) / np.log10(2)                       # shift per doubling, in units of log10(2)
print('tau_star     :', taus)
print('knee time    :', np.round(knee_t, 3), 'Gyr')
print('knee [Fe/H]  :', np.round(knee_feh, 3))
print('shift per doubling / log10(2):', np.round(step, 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cols = plt.cm.viridis(np.linspace(0, 0.9, len(taus)))
for (fe, si), ts, col, kf in zip(tracks, taus, cols, knee_feh):
    axes[0].plot(fe[1:], si[1:], color=col, lw=2, label=fr'$\tau_*$ = {ts:g} Gyr')
    axes[0].plot(kf, plateau['si'] - 0.05, 'o', color=col)
axes[0].set(xlim=(-3, 0.3), xlabel='[Fe/H]', ylabel='[Si/Fe]')
axes[0].legend(frameon=False, fontsize=8)
axes[1].plot(np.log10(taus), knee_feh, 'o-', color='#2a78d6', lw=2, label='model')
axes[1].plot(np.log10(taus), knee_feh[-1] - (np.log10(taus) - np.log10(taus[-1])), 'k--', lw=1, label='slope −1')
axes[1].set(xlabel=r'$\log_{10}(\tau_*/{\rm Gyr})$', ylabel='[Fe/H] of the knee')
axes[1].legend(frameon=False)
for ax in axes:
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()

assert np.ptp(knee_t) < 0.05                                # the knee time hardly depends on tau_star
assert abs(step[-1] + 1) < 0.05                              # controlled experiment: shift -> -log10(2) per doubling
assert np.all(np.diff(step) < 0)                             # ... approached from above as tau_dep grows

**Figure 2.4.** Left: [Si/Fe] against [Fe/H] for constant star formation with no wind ($\eta$ = 0) and five depletion
times $\tau_\star$ = 1–16 Gyr (dark to light); points: the knees, 0.05 dex below the plateau. Right: [Fe/H] of the knee
against $\log_{10}\tau_\star$ (blue); dashed: a line of slope −1 through the last point.

The knee occurs at 0.55–0.58 Gyr in all five galaxies, but at [Fe/H] from −0.3 to −1.4. Each doubling of
$\tau_\star$ moves it by −0.26, −0.28, −0.29 and −0.30 dex in turn (in units of $\log_{10}2$: printed above), approaching the
predicted −0.30 as the depletion time grows long compared with the knee time. **The [Fe/H] of the knee measures the
star-formation efficiency.** This is why the dwarf galaxies of Chapter 0, whose tracks turn down at [Fe/H] ≈ −1.5
or below, are thought to have formed stars slowly, and why GSE, still α-rich at [Fe/H] ≈ −1, formed them fastest.

A stronger wind does not move the knee much (try it: Exercise 1), but it lowers the metallicity where the gas
saturates, and therefore where the decline ends.

## 2.5 The delay-time distribution matters too

In [ ]:
knees = {}
fig, ax = plt.subplots(figsize=(7, 4.3))
for name, f, ls in (('exponential (H21)', dtd_exp, '-'), (r'power law $\tau^{-1.1}$', dtd_plaw, '--')):
    t, Mg, sfr, feh, xfe = constant_sfr(tau_star=4.0, eta=2.5, dtd=f)
    k = np.argmax(xfe['si'] < plateau['si'] - 0.05)
    ax.plot(feh[1:], xfe['si'][1:], color='#1baf7a', lw=2, ls=ls, label=f'{name}: knee at [Fe/H] = {feh[k]:.2f}')
    print(f'{name}: knee at t = {t[k]:.3f} Gyr, [Fe/H] = {feh[k]:.3f};  [Si/Fe] at 13.7 Gyr = {xfe["si"][-1]:.3f}')
    knees[name] = (t[k], feh[k])
ax.set(xlim=(-3, 0.0), xlabel='[Fe/H]', ylabel='[Si/Fe]')
ax.legend(frameon=False, fontsize=9)
ax.grid(color='0.9', lw=0.6)
(te, fe), (tp, fp) = knees.values()
assert tp < te and fp < fe                                  # the power law's knee comes earlier and lower

**Figure 2.5.** [Si/Fe] against [Fe/H] for constant star formation ($\tau_\star$ = 4 Gyr, $\eta$ = 2.5) with the
exponential DTD (solid) and the power-law DTD (dashed); the legend gives the [Fe/H] of each knee.

The power law's earlier explosions put its knee earlier (0.38 against 0.54 Gyr) and at lower [Fe/H] (−1.03 against
−0.91); at 13.7 Gyr the two give nearly the same [Si/Fe]. So the knee's
position measures the star-formation efficiency only *for a given DTD*: the two are
degenerate, and a fit to one galaxy cannot separate them. Comparing galaxies with the same DTD can.

## Summary

- The plateau is the CCSN yield ratio: [Mg/Fe] = +0.26, [Si/Fe] = +0.23 for H21's yields. The model reproduces it
  exactly before the first SN Ia.
- The SN Ia rate is the star-formation history convolved with the DTD; with constant star formation, [α/Fe] tends to
  the ratio of total yields.
- For a given DTD the knee occurs at a fixed time, and its [Fe/H] falls by log₁₀ 2 per doubling of $\tau_\star$: the
  knee measures star-formation efficiency.
- The DTD shape moves the knee too; DTD and efficiency are degenerate in a single galaxy.

## Exercises

1. Repeat Figure 2.4 with $\eta$ = 5. How much does the knee move, and why does the decline now end at lower [Fe/H]?
2. Give the galaxy an exponentially declining star formation, by setting the inflow to zero and starting with gas.
   What happens to [Si/Fe] at late times, and why does it fall below the late-time value of constant star formation?
3. Replace the yields at Z = 0.002 by those at Z = 0.0001 and Z = 0.014 (Figure 2.1). How much do the plateaus move?